In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import joblib

In [2]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_columns(df)

    numeric_cols = [
        "year", "quarter", "units_reimbursed", "number_of_prescriptions",
        "medicaid_amount_reimbursed", "total_amount_reimbursed",
        "non_medicaid_amount_reimbursed", "cost_per_unit",
        "units_per_prescription", "strength"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "cost_per_unit" not in df.columns and {"medicaid_amount_reimbursed", "units_reimbursed"}.issubset(df.columns):
        df["cost_per_unit"] = np.where(
            df["units_reimbursed"] > 0,
            df["medicaid_amount_reimbursed"] / df["units_reimbursed"],
            np.nan
        )

    if "units_per_prescription" not in df.columns and {"units_reimbursed", "number_of_prescriptions"}.issubset(df.columns):
        df["units_per_prescription"] = np.where(
            df["number_of_prescriptions"] > 0,
            df["units_reimbursed"] / df["number_of_prescriptions"],
            np.nan
        )

    expansion_states = {
        "AK", "AZ", "AR", "CA", "CO", "CT", "DC", "DE", "HI", "IA", "ID",
        "IL", "IN", "KS", "KY", "LA", "MA", "MD", "ME", "MI", "MN", "MO",
        "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK",
        "OR", "PA", "RI", "SD", "UT", "VA", "VT", "WA", "WV"
    }

    if "is_expansion_state" not in df.columns and "state" in df.columns:
        df["is_expansion_state"] = np.where(
            df["state"].astype(str).str.strip().isin(expansion_states),
            "Yes",
            "No"
        )
    elif "is_expansion_state" in df.columns:
        df["is_expansion_state"] = df["is_expansion_state"].astype(str).str.strip()

    for col in [
        "state", "therapeutic_class", "epc_class", "routename",
        "labelername", "marketingcategoryname", "productid",
        "ndc", "ndc_standardized", "product_name"
    ]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    return df


def load_dashboard_data():
   
    if "df_model" in globals():
        print("Using df_model from notebook memory.")
        return prepare_dataframe(df_model.copy()), "df_model (memory)"

    # 2. Fallback CSV files
    candidates = [
        "df_model.csv",
        "historical_medicaid_cleaned.csv",
        "historical_analysis_output.csv",
        "medicaid_historical.csv"
    ]

    for file_name in candidates:
        path = Path(file_name)
        if path.exists():
            print(f"Loaded data from {file_name}")
            df = pd.read_csv(path, low_memory=False)
            return prepare_dataframe(df), file_name

    raise FileNotFoundError(
        "No data found. Either keep df_model in memory or save df_model.to_csv('df_model.csv', index=False)."
    )


df_dash, source_name = load_dashboard_data()
print("Source:", source_name)
print("Shape:", df_dash.shape)

Loaded data from df_model.csv
Source: df_model.csv
Shape: (3777265, 22)


In [3]:
def load_prediction_model():
    candidates = [
        "lgbm_medicaid_model.pkl",
        "lgbm_model.pkl"
    ]

    for file_name in candidates:
        path = Path(file_name)
        if path.exists():
            print(f"Loaded model from {file_name}")
            return joblib.load(path), file_name

    raise FileNotFoundError(
        "No model file found. Save your trained model as lgbm_medicaid_model.pkl or lgbm_model.pkl."
    )


model, model_name = load_prediction_model()
print("Model source:", model_name)

Loaded model from lgbm_medicaid_model.pkl
Model source: lgbm_medicaid_model.pkl


In [4]:

drug_id_candidates = ["productid", "ndc_standardized", "ndc", "product_name"]

drug_id_col = None
for c in drug_id_candidates:
    if c in df_dash.columns:
        drug_id_col = c
        break

if drug_id_col is None:
    raise ValueError("No drug identifier column found. Need one of: productid, ndc_standardized, ndc, product_name")

print("Drug identifier column being used:", drug_id_col)

base_model_feature_cols = [
    "state",
    "quarter",
    "units_reimbursed",
    "number_of_prescriptions",
    "units_per_prescription",
    "therapeutic_class",
    "epc_class",
    "routename",
    "labelername",
    "marketingcategoryname",
    "is_expansion_state"
]
if "year" in df_dash.columns:
    model_feature_cols = ["year"] + base_model_feature_cols
else:
    model_feature_cols = base_model_feature_cols
missing_features = [c for c in model_feature_cols if c not in df_dash.columns]
if missing_features:
    raise ValueError(f"Missing required model features: {missing_features}")


def most_common(series, default="Unknown"):
    s = series.dropna().astype(str)
    if s.empty:
        return default
    return s.mode().iloc[0]


def get_next_year_quarter(df):
    if "year" in df.columns and "quarter" in df.columns:
        latest_year = int(df["year"].dropna().max())
        latest_quarter = int(df.loc[df["year"] == latest_year, "quarter"].dropna().max())
        if latest_quarter < 4:
            return latest_year, latest_quarter + 1
        return latest_year + 1, 1
    return 2026, 1


def build_prediction_input(df, selected_drug, selected_state, selected_year, selected_quarter):
    subset = df[
        (df[drug_id_col].astype(str) == str(selected_drug)) &
        (df["state"].astype(str) == str(selected_state))
    ].copy()

    if subset.empty:
        subset = df[df[drug_id_col].astype(str) == str(selected_drug)].copy()

    if subset.empty:
        raise ValueError("No historical rows found for this selected drug/NDC.")

    sort_cols = [c for c in ["year", "quarter"] if c in subset.columns]
    if sort_cols:
        subset = subset.sort_values(sort_cols)

    recent = subset.tail(min(4, len(subset))).copy()
    latest = subset.iloc[-1]

    est_units = recent["units_reimbursed"].median()
    est_rx = recent["number_of_prescriptions"].median()

    if pd.isna(est_units) or est_units <= 0:
        est_units = latest["units_reimbursed"]

    if pd.isna(est_rx) or est_rx <= 0:
        est_rx = latest["number_of_prescriptions"]

    est_units = float(est_units)
    est_rx = float(est_rx)
    est_units_per_rx = est_units / est_rx if est_rx > 0 else np.nan

    row_dict = {
        "state": str(selected_state),
        "quarter": int(selected_quarter),
        "units_reimbursed": est_units,
        "number_of_prescriptions": est_rx,
        "units_per_prescription": est_units_per_rx,
        "therapeutic_class": most_common(subset["therapeutic_class"]),
        "epc_class": most_common(subset["epc_class"]),
        "routename": most_common(subset["routename"]),
        "labelername": most_common(subset["labelername"]),
        "marketingcategoryname": most_common(subset["marketingcategoryname"]),
        "is_expansion_state": "Yes" if str(selected_state).strip() in {
            "AK", "AZ", "AR", "CA", "CO", "CT", "DC", "DE", "HI", "IA", "ID",
            "IL", "IN", "KS", "KY", "LA", "MA", "MD", "ME", "MI", "MN", "MO",
            "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK",
            "OR", "PA", "RI", "SD", "UT", "VA", "VT", "WA", "WV"
        } else "No"
    }

    if "year" in model_feature_cols:
        row_dict["year"] = int(selected_year)

    pred_row = pd.DataFrame([row_dict])

    cat_cols = [
        "state", "therapeutic_class", "epc_class", "routename",
        "labelername", "marketingcategoryname", "is_expansion_state"
    ]

    for col in cat_cols:
        if col in pred_row.columns:
            pred_row[col] = pred_row[col].astype("category")

    return pred_row[model_feature_cols], recent

Drug identifier column being used: productid


In [5]:
def weighted_mean(frame, value_col, weight_col):
    temp = frame[[value_col, weight_col]].dropna()
    temp = temp[temp[weight_col] > 0]
    if temp.empty:
        return np.nan
    return np.average(temp[value_col], weights=temp[weight_col])


def filter_data(df, years, quarters, states, classes, expansion_values, trim_outliers):
    out = df.copy()

    if years and "year" in out.columns:
        out = out[out["year"].isin(years)]

    if quarters and "quarter" in out.columns:
        out = out[out["quarter"].isin(quarters)]

    if states and "state" in out.columns:
        out = out[out["state"].isin(states)]

    if classes and "therapeutic_class" in out.columns:
        out = out[out["therapeutic_class"].isin(classes)]

    if expansion_values and "is_expansion_state" in out.columns:
        out = out[out["is_expansion_state"].isin(expansion_values)]

    if trim_outliers and "cost_per_unit" in out.columns:
        lower = out["cost_per_unit"].quantile(0.01)
        upper = out["cost_per_unit"].quantile(0.99)
        out = out[(out["cost_per_unit"] >= lower) & (out["cost_per_unit"] <= upper)]

    return out

In [6]:
years = sorted(df_dash["year"].dropna().astype(int).unique().tolist()) if "year" in df_dash.columns else []
quarters = sorted(df_dash["quarter"].dropna().astype(int).unique().tolist()) if "quarter" in df_dash.columns else []
states = sorted(df_dash["state"].dropna().astype(str).unique().tolist()) if "state" in df_dash.columns else []
classes = sorted(df_dash["therapeutic_class"].dropna().astype(str).unique().tolist()) if "therapeutic_class" in df_dash.columns else []
expansion_values = sorted(df_dash["is_expansion_state"].dropna().astype(str).unique().tolist()) if "is_expansion_state" in df_dash.columns else []
drug_ids = sorted(df_dash[drug_id_col].dropna().astype(str).unique().tolist())

default_year, default_quarter = get_next_year_quarter(df_dash)

# Prediction widgets
pred_state_widget = widgets.Dropdown(
    options=states,
    description="State:",
    layout=widgets.Layout(width="240px")
)

pred_drug_widget = widgets.Combobox(
    options=drug_ids,
    placeholder="Enter ProductID / NDC",
    description="Drug NDC:",
    ensure_option=True,
    layout=widgets.Layout(width="420px")
)

pred_year_widget = widgets.IntText(
    value=default_year,
    description="Year:",
    layout=widgets.Layout(width="180px")
)

pred_quarter_widget = widgets.Dropdown(
    options=[1, 2, 3, 4],
    value=default_quarter,
    description="Quarter:",
    layout=widgets.Layout(width="180px")
)

predict_button = widgets.Button(
    description="Predict Cost / Unit",
    button_style="success",
    icon="line-chart"
)


year_widget = widgets.SelectMultiple(
    options=years,
    value=tuple(years),
    description="Years",
    layout=widgets.Layout(width="220px", height="160px")
)

quarter_widget = widgets.SelectMultiple(
    options=quarters,
    value=tuple(quarters),
    description="Quarter",
    layout=widgets.Layout(width="180px", height="160px")
)

state_widget = widgets.SelectMultiple(
    options=states,
    value=tuple(states[:12] if len(states) > 12 else states),
    description="States",
    layout=widgets.Layout(width="220px", height="160px")
)

class_widget = widgets.SelectMultiple(
    options=classes,
    value=(),
    description="Class",
    layout=widgets.Layout(width="260px", height="160px")
)

expansion_widget = widgets.SelectMultiple(
    options=expansion_values,
    value=tuple(expansion_values),
    description="Expansion",
    layout=widgets.Layout(width="180px", height="160px")
)

trim_widget = widgets.Checkbox(
    value=True,
    description="Trim top/bottom 1% outliers"
)

metric_widget = widgets.ToggleButtons(
    options=[
        ("Weighted Cost / Unit", "weighted"),
        ("Average Cost / Unit", "average"),
        ("Total Reimbursed", "reimb")
    ],
    description="State Metric:"
)

prediction_out = widgets.Output()
dashboard_out = widgets.Output()

In [7]:
def on_predict_clicked(b):
    with prediction_out:
        clear_output(wait=True)

        selected_drug = pred_drug_widget.value
        selected_state = pred_state_widget.value
        selected_year = pred_year_widget.value
        selected_quarter = pred_quarter_widget.value

        if not selected_drug:
            display(HTML("""
            <div style="background:#fef2f2; border:1px solid #fecaca; color:#991b1b;
                        padding:12px 16px; border-radius:12px;">
                Please enter a valid Drug NDC / ProductID first.
            </div>
            """))
            return

        try:
            X_pred, recent_history = build_prediction_input(
                df_dash,
                selected_drug=selected_drug,
                selected_state=selected_state,
                selected_year=selected_year,
                selected_quarter=selected_quarter
            )

            pred_log = model.predict(X_pred)
            pred_cost_per_unit = float(np.expm1(pred_log[0]))

            est_units = float(X_pred.loc[0, "units_reimbursed"])
            implied_total = pred_cost_per_unit * est_units

            display(HTML(f"""
            <div style="
                background: linear-gradient(135deg, #7c3aed 0%, #ec4899 55%, #06b6d4 100%);
                color: white;
                padding: 18px 22px;
                border-radius: 18px;
                margin-bottom: 14px;">
                <h2 style="margin:0;">🔮 Next Quarter Drug Cost Prediction</h2>
                <p style="margin:8px 0 0 0;">
                    For <b>{selected_drug}</b> in <b>{selected_state}</b>, expected Medicaid
                    <b>cost per unit</b> for <b>{selected_year} Q{selected_quarter}</b> is:
                </p>
                <h1 style="margin:12px 0 0 0;">${pred_cost_per_unit:,.2f}</h1>
            </div>
            """))

            display(HTML(f"""
            <div style="
                background:#f8fafc;
                border:1px solid #e2e8f0;
                padding:14px 18px;
                border-radius:14px;
                margin-bottom:14px;">
                <h3 style="margin-top:0;">Prediction details</h3>
                <ul>
                    <li><b>Estimated Units Reimbursed used:</b> {est_units:,.2f}</li>
                    <li><b>Estimated implied Medicaid reimbursement:</b> ${implied_total:,.2f}</li>
                    <li><b>Therapeutic Class:</b> {X_pred.loc[0, 'therapeutic_class']}</li>
                    <li><b>Route:</b> {X_pred.loc[0, 'routename']}</li>
                    <li><b>Labeler:</b> {X_pred.loc[0, 'labelername']}</li>
                    <li><b>Expansion State:</b> {X_pred.loc[0, 'is_expansion_state']}</li>
                </ul>
            </div>
            """))

            history_cols = [c for c in [
                "year", "quarter", drug_id_col, "state", "units_reimbursed",
                "number_of_prescriptions", "units_per_prescription",
                "cost_per_unit", "medicaid_amount_reimbursed"
            ] if c in recent_history.columns]

            display(HTML("<h3>Recent historical context used</h3>"))
            display(recent_history[history_cols].tail(8))

        except Exception as e:
            display(HTML(f"""
            <div style="
                background:#fef2f2;
                border:1px solid #fecaca;
                color:#991b1b;
                padding:12px 16px;
                border-radius:12px;">
                <b>Prediction failed:</b> {e}
            </div>
            """))

predict_button.on_click(on_predict_clicked)

In [10]:
def render_dashboard(change=None):
    with dashboard_out:
        clear_output(wait=True)

        filtered = filter_data(
            df=df_dash,
            years=list(year_widget.value),
            quarters=list(quarter_widget.value),
            states=list(state_widget.value),
            classes=list(class_widget.value),
            expansion_values=list(expansion_widget.value),
            trim_outliers=trim_widget.value
        )

        if filtered.empty:
            display(HTML("<h3 style='color:#b91c1c;'>No rows match the selected filters.</h3>"))
            return

        avg_cpu = filtered["cost_per_unit"].mean() if "cost_per_unit" in filtered.columns else np.nan
        total_reimb = filtered["medicaid_amount_reimbursed"].sum() if "medicaid_amount_reimbursed" in filtered.columns else np.nan

        display(HTML(f"""
        <div style="
            background: linear-gradient(135deg, #0f766e 0%, #0284c7 55%, #7c3aed 100%);
            color: white;
            padding: 18px 22px;
            border-radius: 18px;
            margin: 12px 0 14px 0;">
            <h1 style="margin:0;">💊 Medicaid Customer Decision Dashboard</h1>
            <p style="margin:8px 0 0 0;">
                Designed from a stakeholder point of view: where reimbursement pressure is high,
                which categories drive cost, and which markets deserve attention first.
            </p>
            <hr style="border: none; border-top: 1px solid rgba(255,255,255,0.35); margin: 14px 0;">
            <h2 style="margin:0 0 8px 0;">What is the expected cost per unit reimbursement for a selected drug next quarter?</h2>
            <p style="margin:0;">
                Enter a <b>state</b> and a <b>drug NDC / ProductID</b>. The model will estimate the expected
                <b>Medicaid cost per unit</b> for the selected next-quarter scenario.
            </p>
        </div>
        """))

        display(pred_controls)
        display(prediction_out)

        display(HTML(f"""
        <div style="display:flex; gap:14px; margin:18px 0 18px 0; flex-wrap:wrap;">
            <div style="background:#ffffff; border:1px solid #dbeafe; padding:12px 16px; border-radius:14px; min-width:180px;">
                <b>Rows</b><br><span style="font-size:24px;">{len(filtered):,}</span>
            </div>
            <div style="background:#ffffff; border:1px solid #dbeafe; padding:12px 16px; border-radius:14px; min-width:180px;">
                <b>States</b><br><span style="font-size:24px;">{filtered['state'].nunique() if 'state' in filtered.columns else 'N/A'}</span>
            </div>
            <div style="background:#ffffff; border:1px solid #dbeafe; padding:12px 16px; border-radius:14px; min-width:180px;">
                <b>Years</b><br><span style="font-size:24px;">{filtered['year'].nunique() if 'year' in filtered.columns else 'N/A'}</span>
            </div>
            <div style="background:#ffffff; border:1px solid #dbeafe; padding:12px 16px; border-radius:14px; min-width:220px;">
                <b>Mean Cost / Unit</b><br><span style="font-size:24px;">${avg_cpu:,.2f}</span>
            </div>
            <div style="background:#ffffff; border:1px solid #dbeafe; padding:12px 16px; border-radius:14px; min-width:240px;">
                <b>Total Medicaid Reimbursed</b><br><span style="font-size:24px;">${total_reimb:,.0f}</span>
            </div>
        </div>
        """))

        if "cost_per_unit" in filtered.columns:
            fig1 = px.histogram(
                filtered.dropna(subset=["cost_per_unit"]),
                x="cost_per_unit",
                nbins=45,
                marginal="box",
                title="Where price pressure is concentrated: Cost per Unit Distribution"
            )
            fig1.update_layout(height=420, xaxis_title="Cost per Unit ($)", yaxis_title="Records")
            fig1.show()

        if {"year", "quarter", "cost_per_unit"}.issubset(filtered.columns):
            trend = (
                filtered.groupby(["year", "quarter"], as_index=False)
                .agg(
                    avg_cost_per_unit=("cost_per_unit", "mean"),
                    total_reimb=("medicaid_amount_reimbursed", "sum") if "medicaid_amount_reimbursed" in filtered.columns else ("cost_per_unit", "size")
                )
                .sort_values(["year", "quarter"])
            )
            trend["period"] = trend["year"].astype(int).astype(str) + " Q" + trend["quarter"].astype(int).astype(str)

            fig2 = px.line(
                trend,
                x="period",
                y="avg_cost_per_unit",
                markers=True,
                title="Customer trend view: Average cost per unit over time"
            )
            fig2.update_layout(height=420, xaxis_title="Period", yaxis_title="Average Cost per Unit ($)")
            fig2.show()

            fig3 = px.bar(
                trend,
                x="period",
                y="total_reimb",
                title="Customer spend view: Total Medicaid reimbursement over time"
            )
            fig3.update_layout(height=420, xaxis_title="Period", yaxis_title="Total Medicaid Reimbursed ($)")
            fig3.show()

        if "state" in filtered.columns:
            if metric_widget.value == "weighted" and {"cost_per_unit", "units_reimbursed"}.issubset(filtered.columns):
                state_summary = (
                    filtered.groupby("state")
                    .apply(lambda g: weighted_mean(g, "cost_per_unit", "units_reimbursed"))
                    .reset_index(name="metric")
                )
                title = "State attention view: Weighted Cost per Unit"
                xlab = "Weighted Cost per Unit ($)"
            elif metric_widget.value == "average" and "cost_per_unit" in filtered.columns:
                state_summary = (
                    filtered.groupby("state", as_index=False)["cost_per_unit"]
                    .mean()
                    .rename(columns={"cost_per_unit": "metric"})
                )
                title = "State attention view: Average Cost per Unit"
                xlab = "Average Cost per Unit ($)"
            else:
                state_summary = (
                    filtered.groupby("state", as_index=False)["medicaid_amount_reimbursed"]
                    .sum()
                    .rename(columns={"medicaid_amount_reimbursed": "metric"})
                )
                title = "State attention view: Total Medicaid Reimbursed"
                xlab = "Total Medicaid Reimbursed ($)"

            state_summary = state_summary.sort_values("metric", ascending=False)

            fig4 = px.bar(
                state_summary.head(20),
                x="metric",
                y="state",
                orientation="h",
                title=title
            )
            fig4.update_layout(height=550, xaxis_title=xlab, yaxis_title="State", yaxis={"categoryorder": "total ascending"})
            fig4.show()

        if {"therapeutic_class", "cost_per_unit"}.issubset(filtered.columns):
            class_summary = (
                filtered.groupby("therapeutic_class", as_index=False)
                .agg(
                    avg_cost=("cost_per_unit", "mean"),
                    reimb=("medicaid_amount_reimbursed", "sum") if "medicaid_amount_reimbursed" in filtered.columns else ("cost_per_unit", "size")
                )
                .sort_values("avg_cost", ascending=False)
                .head(15)
            )

            fig5 = px.bar(
                class_summary,
                x="avg_cost",
                y="therapeutic_class",
                orientation="h",
                title="Category driver view: Highest cost therapeutic classes"
            )
            fig5.update_layout(height=550, xaxis_title="Average Cost per Unit ($)", yaxis_title="Therapeutic Class", yaxis={"categoryorder": "total ascending"})
            fig5.show()

        if {"labelername", "medicaid_amount_reimbursed"}.issubset(filtered.columns):
            labeler_summary = (
                filtered.groupby("labelername", as_index=False)["medicaid_amount_reimbursed"]
                .sum()
                .sort_values("medicaid_amount_reimbursed", ascending=False)
                .head(12)
            )

            fig6 = px.treemap(
                labeler_summary,
                path=["labelername"],
                values="medicaid_amount_reimbursed",
                title="Supplier concentration view: Largest labelers by reimbursement"
            )
            fig6.update_layout(height=500)
            fig6.show()

        

In [9]:
for w in [year_widget, quarter_widget, state_widget, class_widget, expansion_widget, trim_widget, metric_widget]:
    w.observe(render_dashboard, names="value")

pred_controls = widgets.HBox([
    widgets.VBox([pred_state_widget, pred_drug_widget]),
    widgets.VBox([pred_year_widget, pred_quarter_widget, predict_button])
])

dash_controls_row1 = widgets.HBox([year_widget, quarter_widget, state_widget])
dash_controls_row2 = widgets.HBox([class_widget, expansion_widget])
dash_controls_row3 = widgets.HBox([trim_widget, metric_widget])

display(dashboard_out)

display(HTML("<hr><h2>Dashboard Controls</h2>"))
display(dash_controls_row1)
display(dash_controls_row2)
display(dash_controls_row3)

render_dashboard()

Output()